# Tracing Agent Runs with LangSmith and OpenTelemetry
Companion notebook for the **Tracing Agent Runs with LangSmith and OpenTelemetry** section of Chapter 9.

A good trace shows the **root run** and the child steps under it -- model calls, tool calls, routing decisions, retries, failures. Each step is a **span**; the full execution is a **trace**.

**LangSmith** gives you a hosted place to inspect and compare runs (grouping by tags/metadata, side-by-side version comparison) -- it needs an account and API key, so this notebook doesn't call it directly. **OpenTelemetry** is what the book gives literal code for, and it needs neither an account nor a network call: spans can be captured entirely in memory, which is what this notebook does.

Set up a tracer that captures finished spans in memory (swap `InMemorySpanExporter` for an OTLP exporter pointed at LangSmith or another backend in production -- the instrumentation code below doesn't change either way):

In [1]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

provider = TracerProvider()
exporter = InMemorySpanExporter()
provider.add_span_processor(SimpleSpanProcessor(exporter))
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("support-agent")

This is the book's `run_agent` example. One fix from the printed text: `tool_call` has to be **nested inside** `agent_run`'s `with` block to actually become its child span (matching "root run and the child steps under it"), and the final `run_span.set_attribute(...)` + `return` need to stay inside that same block too.

In [2]:
def run_agent(user_input: str):
    with tracer.start_as_current_span("agent_run") as run_span:
        run_span.set_attribute("app.release_id", "2026.04.1")
        run_span.set_attribute("app.prompt_version", "refund_v3")
        run_span.set_attribute("app.prompt_text", "You are a support agent for refunds.")
        run_span.set_attribute("app.model", "gpt-4.1")

        with tracer.start_as_current_span("tool_call") as tool_span:
            tool_span.set_attribute("app.tool_name", "lookup_order")
            tool_span.set_attribute("app.tool_input.order_id", "A123")
            tool_result = {"status": "shipped"}

        run_span.set_attribute("app.tool_output.status", tool_result["status"])
        return tool_result

In [3]:
result = run_agent("Where is my order A123?")
print("result:", result)

result: {'status': 'shipped'}


Inspect what was actually captured -- a real root span plus a real nested child span, sharing one trace ID:

In [4]:
spans = exporter.get_finished_spans()

for span in spans:
    parent_id = f"{span.parent.span_id:x}" if span.parent else "None (root span)"
    print(f"span={span.name!r}  trace_id={span.context.trace_id:x}  span_id={span.context.span_id:x}  parent={parent_id}")
    for key, value in span.attributes.items():
        print(f"    {key} = {value}")

span='tool_call'  trace_id=f9d3f8d593c659eaa4f636f14af263bb  span_id=25b84fa098062f67  parent=45a7226ae3daef8c
    app.tool_name = lookup_order
    app.tool_input.order_id = A123
span='agent_run'  trace_id=f9d3f8d593c659eaa4f636f14af263bb  span_id=45a7226ae3daef8c  parent=None (root span)
    app.release_id = 2026.04.1
    app.prompt_version = refund_v3
    app.prompt_text = You are a support agent for refunds.
    app.model = gpt-4.1
    app.tool_output.status = shipped


This is enough to answer the book's example questions directly from the trace: *which prompt version was used* (`app.prompt_version`), *which tool ran* (`app.tool_name`), and *whether state changed at the right step* (`app.tool_output.status` only appears after the tool span closes). Attach metadata like release ID, prompt version, tool name, and workflow/tenant ID on every span -- an unlabeled trace is hard to filter or compare later.